# Milestone 2: direct-grounding training budget

Test whether twice the update budget improves circle/square grounding on the **same 128-layout corpus**. Train fresh seed-0 weights for **5,760 updates / 184,320 QA presentations**. Keep rendering, direct questions, model, R=2, optimizer and validation records fixed.

Enable GPU and internet, attach `milestone2_geometry_diversity_artifacts.zip` (or its extracted directory), and set `REFERENCE_SOURCE`. Push the implementation and select a committed `REPO_REF` before running. This uses one GPU (`cuda:0`) and preserves Kaggle's PyTorch installation.

Each QA receives three or four presentations, averaging 3⅓. The final checkpoint is evaluated on training and validation only; no test inference, early stopping, automatic extension, or reference-weight initialization. Fresh output directories are required; resume is unsupported.

See `docs/milestones/milestone2_training_budget.md`. The reference is the completed 2,880-update geometry-diversity run. One seed and four validation layouts limit generalization claims.


In [ ]:
REPO_URL = "https://github.com/Krailon/multi-modal-loop-llm.git"
REPO_REF = "milestone2"  # A committed revision containing this notebook; commit SHA preferred.
REPO_DIR = "/kaggle/working/multi-modal-loop-training-budget"
RUN_ROOT = "/kaggle/working/milestone2_training_budget"
REFERENCE_SOURCE = (
    "/kaggle/input/REPLACE_WITH_YOUR_ARTIFACT_PATH/milestone2_geometry_diversity_artifacts.zip"
)

## Checkout and install

Run all cells in order. The resolved revision is recorded with the artifacts.


In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(REPO_DIR).resolve()
RUN_ROOT = Path(RUN_ROOT).resolve()
if REPO_DIR == RUN_ROOT or REPO_DIR.is_relative_to(RUN_ROOT) or RUN_ROOT.is_relative_to(REPO_DIR):
    raise ValueError("Repository and artifacts must use separate directories")
if not REPO_REF or REPO_REF.startswith("-"):
    raise ValueError("Set REPO_REF to a branch, tag, or commit")


def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()


if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
else:
    if git("remote", "get-url", "origin") != REPO_URL:
        raise ValueError("Existing checkout belongs to a different repository")
    if git("status", "--porcelain"):
        raise ValueError("Existing checkout has local changes; use a clean committed checkout")
    subprocess.run(["git", "fetch", "origin", REPO_REF], cwd=REPO_DIR, check=True)
    if git("rev-parse", "HEAD") != git("rev-parse", "FETCH_HEAD"):
        raise ValueError(
            "Existing checkout has a different revision. Set REPO_REF to its recorded commit "
            "to resume, or use a new REPO_DIR for a new run."
        )

resolved_revision = git("rev-parse", "HEAD")
identity = (str(REPO_DIR), resolved_revision)
if globals().get("_notebook_code_identity", identity) != identity:
    raise RuntimeError(
        "The kernel previously loaded another revision; restart it before proceeding"
    )
_notebook_code_identity = identity
print("Resolved code revision:", resolved_revision)
torch_version = importlib.metadata.version("torch")
# No torch extra, requirements.txt, or accelerator replacement.
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)], check=True)
if importlib.metadata.version("torch") != torch_version:
    raise RuntimeError("PyTorch changed during installation; inspect the environment")
# Editable-install .pth files are processed at interpreter startup. Make this
# checkout importable immediately in the running notebook kernel too.
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
# Training/evaluation run in subprocesses using this kernel's Python interpreter.

## Audit the reference and record the protocol

Verify the completed geometry-diversity checkpoint, manifest, reports, settings and derivation against their recorded identities. Reference weights are read only for auditing. Copy the exact corpus and calculate the new presentation budget; no corpus regeneration or held-out changes.


In [ ]:
import multimodal_loop.eval.kaggle_training_budget as helpers
from multimodal_loop.eval.kaggle_training_budget import (
    archive_training_budget,
    prepare_training_budget,
    run_training_budget,
)

if not Path(helpers.__file__).resolve().is_relative_to(REPO_DIR / "src"):
    raise RuntimeError("Another package checkout is cached; restart the kernel")
run = prepare_training_budget(REPO_DIR, RUN_ROOT, REFERENCE_SOURCE)

## Train and evaluate

Run three full passes plus 576 batches of pass four. Validate every 288 updates; use the final step-5,760 checkpoint. Diagnose all 55,296 training and 1,728 validation questions, with original/added training layouts reported separately. Controls retain recipient targets. Logs stream below and are retained.


In [ ]:
report = run_training_budget(run)

## Inspect the results

Training criteria: at least 95% accuracy for each shape. Held-out direct grounding: at least 90% for each shape and 30 percentage-point overall gaps against blank images and both mean shuffle controls. These are diagnostic criteria, not Milestone 2 completion gates.


In [ ]:
import json

from IPython.display import HTML, FileLink, display

for split, metrics in report["splits"].items():
    print(split, {k: metrics[k] for k in ("total", "accuracy", "loss", "all_three")})
    print("Shape accuracy:", metrics["breakdowns"]["shape"])
print(
    "Training subsets:",
    {
        name: {k: m[k] for k in ("total", "accuracy", "loss")}
        for name, m in report["training_geometry_subsets"].items()
    },
)
print(json.dumps(report["assessment"], indent=2))
comparison = json.loads((run.root / "diagnosis" / "comparison.json").read_text())
print(json.dumps(comparison, indent=2))
for split, metrics in report["splits"].items():
    print(split, "Circle/square pair:", metrics["circle_square_pair"])
    print(split, "Identical circle/square predictions:", metrics["circle_square_same_prediction"])
presentations = json.loads((run.root / "training" / "presentations.json").read_text())
print("Presentations:", presentations["qa_visit_histogram"])
display(HTML((run.root / "diagnosis" / "inspection.html").read_text()))

## Retain artifacts

Download the archive for review. It contains the fresh checkpoint, exact corpus, exposure counts, history, predictions, controls, comparison, error preview, provenance and logs. Reference weights are excluded. Differences compare identical corpora against the completed geometry-diversity run. More triangle accuracy alone does not resolve circle/square grounding.


In [ ]:
archive = archive_training_budget(run)
print("Training-budget archive:", archive)
display(FileLink(str(archive)))